In [1]:
import numpy as np
import pandas as pd
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
# Get data into dataframe
df = pd.read_csv('WMA_fractions_v2_with_PV_GMM_ensemble.csv')

In [ ]:
"""
SOM classification of Conservative Temperature, Absolute Salinity and PV.
Scores k = 2..10 using BIC only (via GaussianMixture on SOM node prototypes).

Requires: numpy, pandas, scikit-learn, minisom, matplotlib
Usage (notebook): load your data into a DataFrame called `df`, then run
this script in a cell. Change N_CLUSTERS below to force a specific k.
"""
import numpy as np
import pandas as pd
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

COLS = ["Conservative_Temperature_[deg_C]", "Absolute_Salinity_[PSU]", "PV"]
K_RANGE = range(2, 11)  # 2..10 inclusive


# --------------------------------------------------------------------------
def suggest_grid(X):
    """
    Heuristic grid size: ~5*sqrt(N) nodes total, square layout.
    Avoids PCA to reduce memory usage. [Inference: simpler than original]
    """
    n_nodes = int(5 * np.sqrt(X.shape[0]))
    side = max(2, int(round(np.sqrt(n_nodes))))
    return side, side


def evaluate_k_bic(W, node_ids, k_range, seed=42, cov_type="full"):
    """
    For each k, fit a GaussianMixture on the SOM node prototypes (W)
    and record BIC. Lower BIC = better fit penalised for complexity.

    NOTE: BIC is computed on the SOM nodes, not the raw samples, so it
    depends on grid size. [Inference]
    """
    rows = []
    for k in k_range:
        if k >= len(W):
            continue
        gmm = GaussianMixture(
            n_components=k, covariance_type=cov_type,
            n_init=5, random_state=seed
        ).fit(W)
        rows.append(dict(k=k, bic=gmm.bic(W)))
    return pd.DataFrame(rows).set_index("k")


def plot_scores(scores, path="cluster_scores.png"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(scores.index, scores["bic"], "o-")
    ax.set_title("BIC (lower is better)")
    ax.set_xlabel("k")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


# --------------------------------------------------------------------------
def run_som(df, cols=COLS, grid=None, sigma=None, learning_rate=0.5,
            n_iter=None, k_range=K_RANGE, n_clusters=None,
            cov_type="full", seed=42):
    """
    df         : DataFrame containing `cols`.
    grid       : (nx, ny); if None, suggest_grid() is used.
    n_clusters : force a number of classes; if None, the BIC-optimal k is used.

    Returns (out_df, som, info). info['scores'] holds the BIC table.
    """
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Columns not found in DataFrame: {missing}")

    data = df.replace([np.inf, -np.inf], np.nan).dropna(subset=list(cols)).copy()
    n_dropped = len(df) - len(data)

    scaler = StandardScaler()
    X = scaler.fit_transform(data[list(cols)].values)

    nx, ny = grid if grid is not None else suggest_grid(X)
    sigma = sigma if sigma is not None else max(nx, ny) / 2.0
    n_iter = n_iter if n_iter is not None else max(10_000, 500 * nx * ny)

    som = MiniSom(nx, ny, X.shape[1], sigma=sigma, learning_rate=learning_rate,
                  neighborhood_function="gaussian", topology="rectangular",
                  random_seed=seed)
    som.pca_weights_init(X)
    som.train(X, n_iter, random_order=True)

    bmus = np.array([som.winner(x) for x in X])
    data["bmu_x"], data["bmu_y"] = bmus[:, 0], bmus[:, 1]
    data["node_id"] = bmus[:, 0] * ny + bmus[:, 1]
    node_ids = data["node_id"].values

    W = som.get_weights().reshape(nx * ny, -1)  # node prototypes (scaled)

    scores = evaluate_k_bic(W, node_ids, k_range, seed, cov_type)
    best_k = int(scores["bic"].idxmin())

    if n_clusters is None:
        n_clusters = best_k

    # Final grouping: GMM on node prototypes
    node_lab = GaussianMixture(
        n_components=n_clusters, covariance_type=cov_type,
        n_init=5, random_state=seed
    ).fit(W).predict(W)
    data["class"] = node_lab[node_ids]

    node_phys = scaler.inverse_transform(W)
    class_centres = pd.DataFrame(
        [node_phys[node_lab == c].mean(axis=0) for c in range(n_clusters)],
        columns=list(cols)
    ).rename_axis("class")

    info = dict(
        grid=(nx, ny), n_iter=n_iter, sigma=sigma,
        n_dropped_rows=n_dropped, n_clusters=n_clusters,
        scores=scores, best_k=best_k,
        quantization_error=som.quantization_error(X),
        topographic_error=som.topographic_error(X),
        class_centres=class_centres, scaler=scaler
    )
    return data, som, info


# --------------------------------------------------------------------------
def summarize(out, info, out_csv="classified.csv", plot_path="cluster_scores.png"):
    """Print diagnostics, save the BIC plot and classified table."""
    pd.set_option("display.width", 140)
    print(f"Rows dropped (NaN/inf): {info['n_dropped_rows']}")
    print(f"Grid: {info['grid']} | iterations: {info['n_iter']}")
    print(f"Quantization error: {info['quantization_error']:.3f} | "
          f"Topographic error: {info['topographic_error']:.3f}\n")
    print(info["scores"].round(3), "\n")
    print(f"Best k by BIC: {info['best_k']}")
    print(f"Used k = {info['n_clusters']}\n")
    print(info["class_centres"])
    print(out["class"].value_counts().sort_index())
    plot_scores(info["scores"], plot_path)
    if out_csv:
        out.to_csv(out_csv, index=False)
    print(f"\nSaved {plot_path}" + (f" and {out_csv}" if out_csv else ""))


# --------------------------------------------------------------------------
N_CLUSTERS = None   # set an integer to force a specific number of classes

if "df" in globals():
    out, som, info = run_som(df, n_clusters=N_CLUSTERS)
    summarize(out, info, out_csv=None)
else:
    print("No DataFrame called `df` found; define it first, or call "
          "run_som(your_dataframe) directly.")